<a href="https://colab.research.google.com/drive/1oCFDI-rKVox9DPNIz7dR_o1z3EgxNl36?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
file_path = '/content/iproperty_cleaned.csv'

In [2]:
import pandas as pd
import numpy as np
import concurrent.futures
import re
import time
import os
import psutil
from IPython.display import display, HTML

In [3]:
df_clean = pd.read_csv(file_path)
display(HTML(f"<b>Dataset Loaded Successfully:</b> {len(df_clean):,} records found."))
display(df_clean.head(3))

,listing_url,property_type,price_rm,price_psf,built_up_sqft,land_area_sqft,furnishing,scraped_at
0,https://www.iproperty.com.my/property/iskandar...,Bungalow/Villa,4900000.0,482.00,7200.0,10166.0,Unfurnished,2026-06-21 20:50:17
1,https://www.iproperty.com.my/property/johor-ba...,Terrace/Link,848000.0,403.81,1600.0,2100.0,Unfurnished,2026-06-21 20:50:17
2,https://www.iproperty.com.my/property/kepong/l...,Terrace/Link,1550000.0,939.39,3600.0,1650.0,Fully Furnished,2026-06-21 20:50:17


## Defining the Architecuture
Since the data is already cleaned of text, our heavy-lifting task will be calculating a new categorical `investment_tier` based on numerical logic.

## **Baseline Model** | Single-Threaded Sequential Core Iteration

In [4]:
def run_baseline_numerical(df):
    df_copy = df.copy()
    investment_categories = []

    for idx, row in df_copy.iterrows():
        price = row['price_rm']
        psf = row['price_psf']

        if pd.isna(price) or pd.isna(psf):
            investment_categories.append("Unknown")
        elif price > 1000000 and psf > 800:
            investment_categories.append("Premium Luxury")
        elif price > 500000 and psf > 400:
            investment_categories.append("Mid-Tier")
        else:
            investment_categories.append("Affordable")

    df_copy['investment_tier'] = investment_categories
    return df_copy

## **Technique 1** | Macro-Parallelism (Process-Based Multiprocessing Loop)

In [5]:
def multi_processing_numerical_kernel(df_shard):
    return run_baseline_numerical(df_shard)

## **Technique 2** | Hybrid Framework (Multiprocessing + Micro-Vectorization)

In [6]:
def hybrid_vectorized_numerical_kernel(df_shard):
    df_copy = df_shard.copy()

    # Fast, vectorized condition masking
    conditions = [
        (df_copy['price_rm'].isna()) | (df_copy['price_psf'].isna()),
        (df_copy['price_rm'] > 1000000) & (df_copy['price_psf'] > 800),
        (df_copy['price_rm'] > 500000) & (df_copy['price_psf'] > 400)
    ]
    choices = ["Unknown", "Premium Luxury", "Mid-Tier"]

    df_copy['investment_tier'] = np.select(conditions, choices, default="Affordable")
    return df_copy

## **Execution & Visual Output**

In [7]:
num_cores = os.cpu_count() or 4
chunk_size = int(np.ceil(len(df_clean) / num_cores))
df_shards = [df_clean.iloc[i * chunk_size : (i + 1) * chunk_size] for i in range(num_cores)]

print("Starting Numerical Pipeline Benchmark...\n")
t0 = time.time()
res_base = run_baseline_numerical(df_clean)
print(f"[Baseline Complete] Latency: {time.time() - t0:.3f} seconds")

t1 = time.time()
with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
    res_t1_blocks = list(executor.map(multi_processing_numerical_kernel, df_shards))
res_t1 = pd.concat(res_t1_blocks, ignore_index=True)
print(f"[Tech 1 Complete] Latency: {time.time() - t1:.3f} seconds")

t2 = time.time()
with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
    res_t2_blocks = list(executor.map(hybrid_vectorized_numerical_kernel, df_shards))
res_t2 = pd.concat(res_t2_blocks, ignore_index=True)
print(f"[Tech 2 Complete] Latency: {time.time() - t2:.3f} seconds\n")

display(HTML("<h4>Investment Tier Distribution</h4>"))
display(pd.DataFrame(res_t2['investment_tier'].value_counts()).rename(columns={'count': 'Number of Properties'}))

Starting Numerical Pipeline Benchmark...

[Baseline Complete] Latency: 7.792 seconds
[Tech 1 Complete] Latency: 4.933 seconds
[Tech 2 Complete] Latency: 0.300 seconds



,Number of Properties
investment_tier,
Mid-Tier,42271
Affordable,36974
Premium Luxury,15759
